# 🎨 Vanilla ComfyUI Colab

A clean, minimal setup for running ComfyUI in Google Colab.

---

## 📦 Install ComfyUI & Dependencies

This will install ComfyUI, all required dependencies, and ComfyUI Manager.

In [1]:
# @title ⚙️ Install ComfyUI & Dependencies { display-mode: "form" }
# @markdown ## 2 · Install ComfyUI & Dependencies
# @markdown
# @markdown Clones ComfyUI and installs the full stack including the **speed trio**:
# @markdown - **xformers** — faster attention, lower VRAM
# @markdown - **triton** — GPU kernel acceleration
# @markdown - **sageattention** — additional attention optimization
# @markdown
# @markdown Also sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to reduce OOM crashes.
import os
from pathlib import Path

WORKSPACE = "/content/ComfyUI"

# ── 0. Install uv ─────────────────────────────────────────────────
print("📦 Installing uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
print("✅ uv ready\n")

# ── 1. Clone / update ComfyUI ─────────────────────────────────────
if not os.path.exists(WORKSPACE):
    print("📥 Cloning ComfyUI...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
    print("✅ Cloned\n")
else:
    print("✅ ComfyUI exists, pulling updates...")
    !cd {WORKSPACE} && git pull -q
    print("✅ Updated\n")

%cd {WORKSPACE}

# ── 2. PyTorch ────────────────────────────────────────────────────
# Skip manual install — let Colab's default PyTorch stay (usually cu121/cu128, stable)
print("⚡ Using Colab's default PyTorch (no manual change needed)...")
# Optional: just print current torch version for info
import torch
print(f"  Current torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

# ── 3. Speed stack ────────────────────────────────────────────────
print("🚀 Installing speed stack...")
!uv pip install --system --upgrade xformers triton sageattention

# ── 4. ComfyUI requirements ───────────────────────────────────────
print("📦 Installing ComfyUI requirements...")
!uv pip install --system -r requirements.txt

# ── 5. Core dependencies ──────────────────────────────────────────
print("📚 Installing core dependencies...")
!uv pip install --system \
    accelerate einops diffusers \
    "safetensors>=0.4.2" \
    aiohttp pyyaml Pillow scipy tqdm psutil \
    "tokenizers>=0.13.3" sentencepiece soundfile \
    "kornia>=0.7.1" spandrel torchsde \
    av albumentations opencv-python \
    onnxruntime-gpu color-matcher \
    comfy_aimdo comfy-kitchen

# ── 6. Transformers / HuggingFace ────────────────────────────────
print("🤗 Installing transformers & huggingface-hub (latest stable)...")
!uv pip install --system --upgrade \
    "transformers>=4.45.0" \
    "huggingface-hub>=0.23.0" \
    hf_transfer

# ── 7. CUDA memory optimization ──────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅ CUDA memory config set\n")

# ── 8. ComfyUI Manager ────────────────────────────────────────────
manager_path = f"{WORKSPACE}/custom_nodes/ComfyUI-Manager"
if not os.path.exists(manager_path):
    print("📥 Installing ComfyUI Manager...")
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager {manager_path}
else:
    print("🔄 Updating ComfyUI Manager...")
    !cd {manager_path} && git pull -q
print("✅ ComfyUI Manager ready\n")

# ── 9. Verify ─────────────────────────────────────────────────────
import importlib.metadata as meta
print("📋 Key package versions:")
for pkg in ["torch", "xformers", "transformers", "huggingface-hub", "safetensors"]:
    try:
        print(f"  ✅ {pkg}: {meta.version(pkg)}")
    except Exception:
        print(f"  ❌ {pkg}: not found")

print("\n🎉 Installation complete! Run the next cell.")


📦 Installing uv...
downloading uv 0.10.6 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
✅ uv ready

📥 Cloning ComfyUI...
✅ Cloned

/content/ComfyUI
⚡ Using Colab's default PyTorch (no manual change needed)...
  Current torch: 2.10.0+cu128 | CUDA available: True
🚀 Installing speed stack...
Using Python 3.12.12 environment at: /usr
Resolved 31 packages in 180ms
Prepared 7 packages in 359ms
Uninstalled 5 packages in 74ms
Installed 7 packages in 23ms
 - cuda-pathfinder==1.3.4
 + cuda-pathfinder==1.4.0
 - filelock==3.24.2
 + filelock==3.24.3
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 - numpy==2.0.2
 + numpy==2.4.2
 + sageattention==1.0.6
 - setuptools==75.2.0
 + setuptools==82.0.0
 + xformers==0.0.35
📦 Installing ComfyUI requirements...
Using Python 3.12.12 environment at: /usr
Resolved 100 packages in 587ms
Prepared 18 packages in 2.23s
Installed 18 packages in 142ms
 + av==16.1.0
 + comfy-aimdo==0.2.2
 + comfy-kitchen==0.2.7
 

In [2]:
# @title 🔧 3 · Custom Nodes { display-mode: "form" }
# @markdown Add GitHub repos to `CUSTOM_NODES` below and uncomment to install.
# @markdown Re-running is safe — existing nodes are skipped automatically.
import os, shutil

# ─── CONFIGURATION ───
WORKSPACE        = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{WORKSPACE}/custom_nodes"

# Add any GitHub URL here. Format: ("Folder_Name", "URL")
CUSTOM_NODES = [
    # Top speed accelerators for Flux.2 / klein / Turbo (install via Manager or git clone)
    ("ComfyUI-CacheDiT", "https://github.com/Jasonzzt/ComfyUI-CacheDiT"),                 # Residual/DiT caching: Strong for Flux.2 klein series (distilled/fast models), simple plug-in accelerator node, good 1.4–1.6× gains with zero extra config.
    ("VHS_VideoCombine", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite"),
]

# ─── INSTALLATION LOGIC ───
print("🚀 Starting Custom Node Installation...\n")

for name, url in CUSTOM_NODES:
    path = os.path.join(CUSTOM_NODES_DIR, name)

    # 1. FIX BROKEN DOWNLOADS
    # If folder exists but isn't a git repo, it's a failed download from a previous run.
    if os.path.exists(path) and not os.path.exists(os.path.join(path, ".git")):
        print(f"⚠️ Found broken folder '{name}', cleaning up...")
        shutil.rmtree(path)

    # 2. CLONE REPO
    if not os.path.exists(path):
        print(f"📥 Cloning: {name}")
        # We use ! instead of os.system to see real-time progress and errors
        !git clone {url} {path}
    else:
        print(f"⏭️  Already exists: {name}")
        # Optional: update existing nodes
        # !cd {path} && git pull

    # 3. INSTALL REQUIREMENTS
    req_file = os.path.join(path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"📦 Installing dependencies for {name}...")
        # We removed -q so you can see if a specific library fails to install
        !uv pip install --system -r {req_file}
    else:
        print(f"ℹ️  No requirements.txt for {name}")

print("\n✨ All custom nodes processed.")

🚀 Starting Custom Node Installation...

📥 Cloning: ComfyUI-CacheDiT
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-CacheDiT'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 166 (delta 48), reused 34 (delta 27), pack-reused 102 (from 1)
Receiving objects: 100% (166/166), 2.55 MiB | 8.21 MiB/s, done.
Resolving deltas: 100% (100/100), done.
📦 Installing dependencies for ComfyUI-CacheDiT...
Using Python 3.12.12 environment at: /usr
Resolved 61 packages in 95ms
Prepared 1 package in 48ms
Installed 1 package in 7ms
 + cache-dit==1.2.3
📥 Cloning: VHS_VideoCombine
Cloning into '/content/ComfyUI/custom_nodes/VHS_VideoCombine'...
remote: Enumerating objects: 3332, done.
remote: Counting objects: 100% (1579/1579), done.
remote: Compressing objects: 100% (359/359), done.
remote: Total 3332 (delta 1449), reused 1220 (delta 1220), pack-reused 1753 (from 3)
Receiving objects: 100% (3332/3332),

In [3]:
# @title 📥 Download Models { display-mode: "form" }
# @markdown Add as many models as you want in the list below.<br>
# @markdown - For **Hugging Face** → use `"type": "hf"` + `repo_id` + optional `filename` (None = full repo).<br>
# @markdown - For **Civitai / mirrors / other** → use `"type": "url"` + direct `url` + `filename`.

import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download
from tqdm import tqdm
import subprocess

# Install aria2c (required for fast URL downloads)
!apt-get update -qq && apt-get install -y -qq aria2
print("✅ aria2c installed\n")

# --- FIXED: Get token from Colab Secrets ---
token = ""
try:
    from google.colab import userdata
    token = userdata.get('CIVITAI_API').strip()
except Exception:
    # Fallback to standard environment variables if not in Colab
    token = os.environ.get("CIVITAI_API", "").strip()

if not token:
    print("⚠️ No CIVITAI_API token found in Colab Secrets. Downloads may fail.")
else:
    print(f"✅ Using Civitai token: {token[:6]}... (hidden)")
# ────────────────────────────────────────────────
#   ↓↓↓  ADD / EDIT YOUR MODELS HERE  ↓↓↓
# ────────────────────────────────────────────────

downloads = [
    {
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors",
    "subfolder": "split_files/diffusion_models",
    "folder": "models/diffusion_models",
},
{
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors",
    "subfolder": "split_files/diffusion_models",
    "folder": "models/diffusion_models",
},
    {
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "wan2.2_t2v_lightx2v_4steps_lora_v1.1_high_noise.safetensors",
    "subfolder": "split_files/loras",
    "folder": "models/loras",
},
{
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "wan2.2_t2v_lightx2v_4steps_lora_v1.1_low_noise.safetensors",
    "subfolder": "split_files/loras",
    "folder": "models/loras",
},
    {
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "subfolder": "split_files/text_encoders",
    "folder": "models/text_encoders",
},
{
    "type": "hf",
    "repo_id": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    "filename": "wan_2.1_vae.safetensors",
    "subfolder": "split_files/vae",
    "folder": "models/vae",
},
    {
    "type": "hf",
    "repo_id": "Lightricks/LTX-2",
    "filename": "ltx-2-19b-dev-fp8.safetensors",
    "subfolder": "",
    "folder": "models/checkpoints",
},
{
    "type": "hf",
    "repo_id": "Comfy-Org/ltx-2",
    "filename": "gemma_3_12B_it_fp4_mixed.safetensors",
    "subfolder": "split_files/text_encoders",
    "folder": "models/text_encoders",
},
{
    "type": "hf",
    "repo_id": "Lightricks/LTX-2",
    "filename": "ltx-2-spatial-upscaler-x2-1.0.safetensors",
    "subfolder": "",
    "folder": "models/latent_upscale_models",
},
# Optional distilled LoRA for even lower VRAM/speed
{
    "type": "hf",
    "repo_id": "Lightricks/LTX-2",
    "filename": "ltx-2-19b-distilled-lora-384.safetensors",
    "subfolder": "",
    "folder": "models/loras",
},
]

# ────────────────────────────────────────────────
#     Usually no need to change anything below
# ────────────────────────────────────────────────

base_dir = "/content/ComfyUI"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def hf_download_flat(repo, filename, subfolder, target_folder):
    """Download from HF and move file flat into target_folder, ignoring subfolder structure."""
    downloaded = hf_hub_download(
        repo_id=repo,
        filename=filename,
        subfolder=subfolder or "",
        local_dir=target_folder,
    )
    final_path = os.path.join(target_folder, os.path.basename(downloaded))
    if os.path.abspath(downloaded) != os.path.abspath(final_path):
        shutil.move(downloaded, final_path)
        try:
            leftover = os.path.dirname(downloaded)
            while leftover != target_folder:
                if not os.listdir(leftover):
                    os.rmdir(leftover)
                leftover = os.path.dirname(leftover)
        except:
            pass
        print(f"   ✅ Saved: {final_path}")
    else:
        print(f"   ✅ Saved: {final_path}")

for item in tqdm(downloads, desc="Downloading models"):
    target_folder = os.path.join(base_dir, item["folder"].lstrip("/"))
    os.makedirs(target_folder, exist_ok=True)

    if item["type"].lower() in["hf", "huggingface"]:
        repo = item["repo_id"]
        file = item.get("filename")
        subfolder = item.get("subfolder", "")
        print(f"\n📥 HF   → {repo}  →  {file or 'FULL REPO'}")

        try:
            if file is None:
                snapshot_download(
                    repo_id=repo,
                    local_dir=target_folder,
                    local_dir_use_symlinks=False,
                )
            else:
                final_path = os.path.join(target_folder, os.path.basename(file))
                if os.path.exists(final_path):
                    print(f"   ⏭️  Already exists, skipping: {final_path}")
                else:
                    hf_download_flat(repo, file, subfolder, target_folder)
        except Exception as e:
            print(f"  ⚠️ Error: {e}")
            print("   → Check repo_id / filename / subfolder")

    elif item["type"].lower() == "url":
        url   = item["url"]
        fname = item["filename"]

        # --- APPEND CIVITAI TOKEN ---
        download_url = url
        if "civitai.com" in url and token:
            sep = "&" if "?" in url else "?"
            download_url = f"{url}{sep}token={token}"

        print(f"\n📥 URL  → {url}  →  {fname}")

        cmd =[
            "aria2c", "--console-log-level=error", "-c",
            "-x", "16", "-s", "16", "-j", "8", "-k", "1M",
            download_url, "-d", target_folder, "-o", fname
        ]
        try:
            subprocess.run(cmd, check=True)
            print(f"   ✅ Saved: {os.path.join(target_folder, fname)}")
        except Exception as e:
            print(f"  ⚠️ aria2c failed: {e}")

    else:
        print(f"⚠️ Unknown type '{item['type']}' — skipping")

print("\n✅ All downloads finished! Check folders in /content/ComfyUI/models/")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/l


📥 HF   → Comfy-Org/Wan_2.2_ComfyUI_Repackaged  →  wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors


split_files/diffusion_models/wan2.2_t2v_(…):   0%|          | 0.00/14.3G [00:00<?, ?B/s]

   ✅ Saved: /content/ComfyUI/models/diffusion_models/wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors

📥 HF   → Comfy-Org/Wan_2.2_ComfyUI_Repackaged  →  wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors


split_files/diffusion_models/wan2.2_t2v_(…):   0%|          | 0.00/14.3G [00:00<?, ?B/s]

...

   ✅ Saved: /content/ComfyUI/models/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors

📥 HF   → Lightricks/LTX-2  →  ltx-2-spatial-upscaler-x2-1.0.safetensors


ltx-2-spatial-upscaler-x2-1.0.safetensor(…):   0%|          | 0.00/996M [00:00<?, ?B/s]

   ✅ Saved: /content/ComfyUI/models/latent_upscale_models/ltx-2-spatial-upscaler-x2-1.0.safetensors

📥 HF   → Lightricks/LTX-2  →  ltx-2-19b-distilled-lora-384.safetensors


ltx-2-19b-distilled-lora-384.safetensors:   0%|          | 0.00/7.67G [00:00<?, ?B/s]

   ✅ Saved: /content/ComfyUI/models/loras/ltx-2-19b-distilled-lora-384.safetensors

✅ All downloads finished! Check folders in /content/ComfyUI/models/


---

# 🚀 Launch ComfyUI

**Run this cell AFTER restarting the runtime.**

In [ ]:
import threading
import time
import socket

def iframe_thread(port):
    """Wait for server to start and display in iframe"""
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()

    from google.colab import output
    print("\n🌐 ComfyUI is ready! Opening in iframe below...")
    print("💡 Tip: Click the link below to open in a new window\n")
    output.serve_kernel_port_as_iframe(port, height=1024)
    output.serve_kernel_port_as_window(port)

# Change to ComfyUI directory
%cd /content/ComfyUI

# Start iframe thread
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🚀 Starting ComfyUI...\n")
print("⏳ Please wait for the interface to load below...\n")

# Launch ComfyUI
!python main.py --dont-print-server

/content/ComfyUI
🚀 Starting ComfyUI...

⏳ Please wait for the interface to load below...

[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-02-26 12:17:46.764
** Platform: Linux
** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   6.5 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Found comfy_kitchen backend triton: {'

<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data: 5/127
FETCH ComfyRegistry Data: 10/127
FETCH ComfyRegistry Data: 15/127
FETCH ComfyRegistry Data: 20/127
[DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/widgetInputs.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui/components/buttonGroup.js. This is likely caused by 

---

## 📝 Notes

- **First time setup**: The installation takes 5-10 minutes
- **Models**: Download models through the ComfyUI interface or manually to `/content/ComfyUI/models/`
- **Custom nodes**: Use ComfyUI Manager (if installed) to add custom nodes
- **Persistence**: Files are stored in the Colab runtime and will be lost when the session ends
- **Google Drive**: Enable the Google Drive option to save workflows and outputs

## 🐛 Troubleshooting

- **403 Error**: Check your browser settings or disable extensions that might block iframes
- **Server won't start**: Restart the runtime and try again
- **Missing models**: Download required models to the appropriate folder in `/content/ComfyUI/models/`

---

**Enjoy using ComfyUI! 🎨**